In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas
import sklearn
import xgboost
import sklearn.linear_model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from datetime import datetime
import random

In [ ]:
def woe(data):
    import math
    large1, large0 = sum(data[data.columns[-1]]), sum(data[data.columns[-1]] == False)
    for column in data.columns[:-1]:
        local = {}
        for unique in set(data[column].unique()):
            local_1, local_0 = sum(data[data[column] == unique][data.columns[-1]]), sum(data[data[column] == unique][data.columns[-1]] == False)
            local[unique] =  (local_1, local_0)
        temp = []
        for row in data[column]:
            small1, small0 = local[row][0], local[row][1]
            if small1 == 0:
                temp.append(0)
            elif small0 != 0:
                temp.append(
                        math.log(
                            (small1/large1)/(small0/large0)
                                )
                )
            else:
                temp.append(
                        math.log(
                            (small1/large1)
                                )
                )
        data[column] = temp
    return data

def process_date(data, column):
    date = pandas.to_datetime(data[column])
    data[column] = (date - date.mean())/date.std()

In [ ]:
hotel = pandas.read_csv("/kaggle/input/hotel-booking/hotel_booking.csv")
hotel["reserved_equal_assigned"]  = hotel["reserved_room_type"] == hotel["assigned_room_type"]
hotel["y"] = hotel["is_canceled"]
hotel = hotel.drop(columns = ["is_canceled", "name", "email", "phone-number", "credit_card", 
                              "reservation_status", "children", "required_car_parking_spaces"])
hotel

In [ ]:
#binning
hotel["company"] = hotel["company"].apply(lambda x:1 if x > 0 else 0)
hotel["agent"] = hotel["agent"].apply(lambda x:1 if x>0 else 0)
hotel["country"] = hotel["country"].apply(lambda x:x if x in ["PRT", "GBR", "FRA", 
                                                              "ESP", "DEU", "ITA", 
                                                              "IRL", "BEL", "BRA", "NLD", "USA"] else "Other")
hotel["country"] = hotel["country"].apply(lambda x:"NLD_FRA" if x in ["NLD", "FRA"] else x)
hotel["country"] = hotel["country"].apply(lambda x:"GBR_BEL" if x in ["GBR", "BEL"] else x)
hotel["country"] = hotel["country"].apply(lambda x:"USA_IRL" if x in ["USA", "IRL"] else x)
hotel["arrival_date_month"] = hotel["arrival_date_month"].apply(lambda x:"Jan_Nov" if x in ["January", "November"] else x)
hotel["arrival_date_month"] = hotel["arrival_date_month"].apply(lambda x:"Jul_Aug_Oct" if x in ["July", "August", "October"] else x)
hotel["arrival_date_month"] = hotel["arrival_date_month"].apply(lambda x:"Apr_Jun" if x in ["April", "June"] else x)
hotel["arrival_date_month"] = hotel["arrival_date_month"].apply(lambda x:"Sep_May" if x in ["September", "May"] else x)

weeks = (
    [3, 6],
    [52, 11],
    [2, 7],
    [53, 35],
    [19, 26],
    [4, 9, 10, 34],
    [15, 42, 38, 45, 37],
    [43, 39],
    [14, 28, 29],
    [41, 30],
    [33, 17],
    [23, 24]
)
for i in weeks:
    hotel["arrival_date_week_number"] = hotel["arrival_date_week_number"].apply(lambda x: i[0] if x in i else x)
    
hotel["stays_in_weekend_nights"] = hotel["stays_in_weekend_nights"].apply(lambda x:2 if x >= 2 else x)
hotel["stays_in_week_nights"] = hotel["stays_in_week_nights"].apply(lambda x:5 if x >= 5 else x)
hotel["adults"] = hotel["adults"].apply(lambda x:3 if x >= 3 else x)
hotel["babies"] = hotel["babies"].apply(lambda x:1 if x > 0 else 0)
hotel["market_segment"] = hotel["market_segment"].apply(lambda x:"Complementary" if x == "Undefined" else x)
hotel["previous_cancellations"] = hotel["previous_cancellations"].apply(lambda x:1 if x > 0 else 0)
hotel["previous_bookings_not_canceled"] = hotel["previous_bookings_not_canceled"].apply(lambda x:1 if x > 0 else 0)
hotel["booking_changes"] = hotel["booking_changes"].apply(lambda x:1 if x > 0 else 0)
hotel["total_of_special_requests"] = hotel["total_of_special_requests"].apply(lambda x:1 if x > 0 else 0)

In [ ]:
#process datetime
process_date(hotel, "reservation_status_date")

In [ ]:
#woe encoding
to_woe = ["hotel", "arrival_date_year", "arrival_date_month", 
          "arrival_date_week_number", "arrival_date_day_of_month", "stays_in_weekend_nights", 
         "stays_in_week_nights", "adults", "babies", 
          "meal", "country", "market_segment", "distribution_channel",
         "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled",
         "reserved_room_type", "assigned_room_type", "booking_changes", "deposit_type",
         "agent", "company", "customer_type", "total_of_special_requests", "reserved_equal_assigned"]
processed = woe(hotel[to_woe + ["y"]])
hotel[to_woe + ["y"]] = processed

In [ ]:
#standardization
hotel[[i for i in hotel.columns[:-1] if i not in to_woe]] = hotel[[i for i in hotel.columns[:-1] if i not in to_woe]].apply(lambda x:(x-x.mean())/x.std())

In [ ]:
#split data
x = hotel[hotel.columns[:-1]]
y = hotel[hotel.columns[-1]]
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size = 0.3, random_state = 2200)

In [ ]:
#LR
lr_cls = sklearn.linear_model.LogisticRegression()
lr_clf = lr_cls.fit(train_x, train_y)

#XGB
xgb_cls = xgboost.XGBClassifier()
xgb_clf = xgb_cls.fit(train_x, train_y)

#prediction, evaluation
lr_predict_y = lr_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(lr_predict_y, test_y)
print("\nLR accuracy: {}\nLR auc: {}\nKS: {}".format(sum(lr_predict_y == test_y)/len(lr_predict_y), 
                                             sklearn.metrics.roc_auc_score(lr_predict_y, test_y),
                                                    max(tpr-fpr)))
xgb_predict_y = xgb_clf.predict(test_x)
fpr, tpr, threshold = sklearn.metrics.roc_curve(xgb_predict_y, test_y)
print("\nXGB accuracy: {}\nXGB auc: {}\nKS: {}".format(sum(xgb_predict_y == test_y)/len(xgb_predict_y), 
                                               sklearn.metrics.roc_auc_score(xgb_predict_y, test_y),
                                              max(tpr-fpr)))